In [ ]:
# ---- Cell 0: Re-extract the dataset (run first in every NEW Colab session)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
archive_path = os.path.join(PROJECT_ROOT, 'data/raw/LJSpeech-1.1.tar.bz2')
print('Archive in Drive:', os.path.exists(archive_path))

if not os.path.isdir('/content/LJSpeech-1.1'):
    !tar -xjf "{archive_path}" -C /content
print('Number of wavs:', len(os.listdir('/content/LJSpeech-1.1/wavs')))  # expect 13100

In [ ]:
# ---- Cell 1: Standard start-of-notebook setup
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
RAW = cfg['abs_paths']['raw']
RESULTS = cfg['abs_paths']['results']
os.makedirs(RESULTS, exist_ok=True)
print('Raw data:', RAW, '| exists:', os.path.isdir(RAW))

In [ ]:
# ---- Cell 2: Load metadata (pipe-separated, no header, quotes must NOT be interpreted)
import pandas as pd, csv

meta = pd.read_csv(
    os.path.join(RAW, 'metadata.csv'),
    sep='|', header=None, names=['id', 'raw_text', 'norm_text'],
    quoting=csv.QUOTE_NONE, encoding='utf-8'
)
print('Rows:', len(meta))
print('Missing norm_text:', meta['norm_text'].isna().sum())
print('Duplicate IDs:', meta['id'].duplicated().sum())
meta.head(10)

In [ ]:
# ---- Cell 3: Raw vs normalized text: where do they differ?
diff = meta[meta['raw_text'] != meta['norm_text']].dropna()
print('Rows where raw != normalized:', len(diff), f'({len(diff)/len(meta):.1%})')
pd.set_option('display.max_colwidth', 120)
diff.sample(8, random_state=0)[['id', 'raw_text', 'norm_text']]

In [ ]:
# ---- Cell 4: Audio properties for ALL files (header read only, fast)
import soundfile as sf
from tqdm import tqdm

rows = []
for cid in tqdm(meta['id']):
    info = sf.info(os.path.join(RAW, 'wavs', f'{cid}.wav'))
    rows.append((cid, info.samplerate, info.channels, info.subtype, info.duration))

audio = pd.DataFrame(rows, columns=['id', 'sr', 'channels', 'subtype', 'duration'])
print(audio['sr'].value_counts())
print(audio['channels'].value_counts())
print(audio['subtype'].value_counts())

In [ ]:
# ---- Cell 5: Duration statistics and histogram
import matplotlib.pyplot as plt

d = audio['duration']
print(f'Total hours : {d.sum()/3600:.2f}')
print(f'Min / Max   : {d.min():.2f} s / {d.max():.2f} s')
print(f'Mean / Median: {d.mean():.2f} s / {d.median():.2f} s')
print(d.describe())

plt.figure(figsize=(7,3.5))
plt.hist(d, bins=60)
plt.xlabel('Clip duration (s)'); plt.ylabel('Count'); plt.title('LJSpeech clip durations')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, '01_clip_duration_hist.png'), dpi=150)
plt.show()

In [ ]:
# ---- Cell 6: Transcript length and speaking rate
df = meta.merge(audio, on='id')
df['norm_text'] = df['norm_text'].fillna(df['raw_text'])   # fall back if norm is missing
df['n_chars'] = df['norm_text'].str.len()
df['n_words'] = df['norm_text'].str.split().str.len()
df['words_per_sec'] = df['n_words'] / df['duration']

print(df[['n_chars', 'n_words', 'words_per_sec']].describe())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(df['n_words'], bins=50);       ax[0].set_title('Words per clip')
ax[1].hist(df['words_per_sec'], bins=50); ax[1].set_title('Speaking rate (words/s)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, '01_text_and_rate_hist.png'), dpi=150)
plt.show()

In [ ]:
# ---- Cell 7: Pick 5 clips to listen to (2 short, 2 long, 1 with digits/quotes in the raw text)
short = df.nsmallest(2, 'duration')['id'].tolist()
long_ = df.nlargest(2, 'duration')['id'].tolist()
odd = df[df['raw_text'].str.contains(r'\d', na=False)].sample(1, random_state=1)['id'].tolist()
picks = short + long_ + odd

import IPython.display as ipd
for cid in picks:
    r = df[df['id'] == cid].iloc[0]
    print(f"\n{cid} | {r['duration']:.2f}s")
    print('RAW :', r['raw_text'])
    print('NORM:', r['norm_text'])
    ipd.display(ipd.Audio(os.path.join(RAW, 'wavs', f'{cid}.wav')))

In [ ]:
# ---- Cell 8: Waveform + mel spectrogram for one clip, with leading/trailing silence marked
import librosa, librosa.display, numpy as np

a = cfg['audio']
cid = picks[2]
y, sr = librosa.load(os.path.join(RAW, 'wavs', f'{cid}.wav'), sr=a['sample_rate'])

_, (start, end) = librosa.effects.trim(y, top_db=30)
print(f'Clip {cid}: lead silence {start/sr:.3f}s | trail silence {(len(y)-end)/sr:.3f}s')

mel = librosa.feature.melspectrogram(
    y=y, sr=sr, n_fft=a['n_fft'], hop_length=a['hop_length'],
    n_mels=a['n_mels'], fmin=a['f_min'], fmax=a['f_max'])
mel_db = librosa.power_to_db(mel, ref=np.max)

fig, ax = plt.subplots(2, 1, figsize=(10, 6))
librosa.display.waveshow(y, sr=sr, ax=ax[0])
ax[0].axvline(start/sr, color='r'); ax[0].axvline(end/sr, color='r'); ax[0].set_title(cid)
librosa.display.specshow(mel_db, sr=sr, hop_length=a['hop_length'], x_axis='time', y_axis='mel', ax=ax[1])
plt.tight_layout()
plt.savefig(os.path.join(RESULTS, '01_wave_and_mel.png'), dpi=150)
plt.show()
print('Frames in this clip:', mel.shape[1], '| expected ~', int(len(y)/a['hop_length'])+1)

In [ ]:
# ---- Cell 9: Alignment risk scan
import re
from collections import Counter

# (a) Characters that are not letters, space or basic punctuation
char_counts = Counter(''.join(df['norm_text']))
odd_chars = {c: n for c, n in char_counts.items() if not re.match(r"[A-Za-z .,;:!?'\-\"()]", c)}
print('Unusual characters:', odd_chars)

# (b) Digits left in the normalized text (these will not be in the dictionary)
digits = df[df['norm_text'].str.contains(r'\d')]
print('Normalized rows still containing digits:', len(digits))

# (c) Vocabulary and rare words
words = Counter(w.lower().strip('.,;:!?"()') for t in df['norm_text'] for w in t.split())
print('Vocabulary size:', len(words))
print('Words seen only once:', sum(1 for w, n in words.items() if n == 1))

# (d) Very short clips and very fast/slow speech
print('Clips < 1.5 s:', (df['duration'] < 1.5).sum())
print('Fastest 5 clips (words/s):')
print(df.nlargest(5, 'words_per_sec')[['id', 'duration', 'words_per_sec']])

In [ ]:
# ---- Cell 10: Save the summary for the report (this is what Notebook 09 will read)
summary = {
    'n_clips': int(len(df)),
    'total_hours': round(float(df['duration'].sum()/3600), 2),
    'mean_duration_s': round(float(df['duration'].mean()), 2),
    'max_duration_s': round(float(df['duration'].max()), 2),
    'sample_rates': {int(k): int(v) for k, v in audio['sr'].value_counts().items()},
    'mean_words_per_sec': round(float(df['words_per_sec'].mean()), 2),
    'vocab_size': len(words),
    'rows_raw_neq_norm': int(len(diff)),
}
import json
with open(os.path.join(RESULTS, '01_dataset_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
df.to_csv(os.path.join(cfg['abs_paths']['processed'], 'metadata_with_stats.csv'), index=False)